# 1. SELECT문 연습

In [ ]:
-- [기본]
-- Q1. 서울에 사는 고객의 이름과 등급 조회

SELECT customer_name, grade
FROM tb_customer
WHERE city='서울';

In [ ]:
-- Q2. 이름이 '이'로 시작하는 고객 조회

SELECT *
FROM tb_customer
WHERE customer_name LIKE '이%';

In [ ]:
-- Q3. 가격이 5,000~50,000원인 상품을 저렴한 순으로 조회

SELECT product_name, unit_price
FROM tb_product
WHERE unit_price >= 5000 AND unit_price <= 50000
ORDER BY unit_price;

In [ ]:
-- Q4. 2024년 2분기(4~6월) 주문 건수 조회

SELECT COUNT(*)
FROM tb_order
WHERE order_dt >= '2024-04-01' AND order_dt < '2024-07-01';

In [ ]:
-- Q5. 등급이 지정되지 않은 고객 조회

SELECT *
FROM tb_customer
WHERE grade IS NULL;

In [ ]:
-- [집계]
-- Q6. 카테고리별 상품 수와 평균 가격 조회 (평균가 높은 순)

SELECT category_id, COUNT(*) AS count, AVG(unit_price) AS average_price
FROM tb_product
GROUP BY category_id
ORDER BY average_price DESC;

In [ ]:
-- Q7. 고객 등급별 인원수와 비율(%) 조회

SELECT grade, COUNT(*) AS count, (COUNT(*) / (SELECT COUNT(*) FROM tb_customer) * 100) AS 비율
FROM tb_customer
GROUP BY grade
ORDER BY 비율 DESC;

In [ ]:
-- Q8. 월별 주문 건수 (취소 제외) 조회 ※ DATE_FORMAT('%Y-%m') 활용

SELECT DATE_FORMAT(order_dt, '%Y-%m') AS month, COUNT(*) AS order_count
FROM tb_order
WHERE status <> 'CANCELED'
GROUP BY DATE_FORMAT(order_dt, '%Y-%m');

In [ ]:
-- Q9. 상품 재고 총합이 100개 미만인 카테고리 조회

SELECT c.category_name, SUM(p.stock_qty) AS 재고총합
FROM tb_product p
    JOIN tb_category c ON p.category_id = c.category_id
GROUP BY p.category_id
HAVING SUM(p.stock_qty) < 100;

In [ ]:
-- Q10. 가장 비싼 상품과 가장 싼 상품의 가격 차이 조회
SELECT MAX(unit_price) - MIN(unit_price) AS 가격차이
FROM tb_product;

In [ ]:
-- [조인]
-- Q11. 주문별 고객명 · 주문일 · 주문금액 합계 조회

SELECT o.order_id, c.customer_name, o.order_dt, SUM(oi.unit_price * oi.qty) AS 합계주문금액
FROM tb_order o
    JOIN tb_customer c ON o.customer_id = c.customer_id
    JOIN tb_order_item oi ON o.order_id = oi.order_id
GROUP BY o.order_id;

In [ ]:
-- Q12. 한 번도 주문하지 않은 고객 조회

SELECT *
FROM tb_customer c
WHERE NOT EXISTS (
    SELECT 1
    FROM tb_order o
    WHERE c.customer_id = o.customer_id
);

In [ ]:
-- Q13. 한 번도 팔리지 않은 상품 조회

SELECT *
FROM tb_product p
WHERE NOT EXISTS (
    SELECT 1
    FROM tb_order_item oi
    WHERE p.product_id = oi.product_id
);

In [ ]:
-- Q14. 고객별 총 구매금액 TOP 5 조회 (취소 제외)

SELECT c.customer_id, c.customer_name, SUM(oi.unit_price * oi.qty) AS 총구매금액
FROM tb_order o
    JOIN tb_customer c ON o.customer_id = c.customer_id
    JOIN tb_order_item oi ON o.order_id = oi.order_id
WHERE o.status <> 'CANCELED'
GROUP BY c.customer_id
ORDER BY 총구매금액 DESC
LIMIT 5;

In [ ]:
-- Q15. 국가별 · 카테고리별 매출 조회

SELECT c.country, cat.category_name, SUM(oi.qty * oi.unit_price) AS 매출
FROM tb_customer c 
    JOIN tb_order o ON c.customer_id = o.customer_id
    JOIN tb_order_item oi ON o.order_id = oi.order_id
    JOIN tb_product p ON oi.product_id = p.product_id
    JOIN tb_category cat ON p.category_id = cat.category_id
GROUP BY c.country, cat.category_name;

In [ ]:
-- [응용 - CASE 피벗]
-- Q16. 카테고리별 상품 수와, 상품 수가 3개 초과면 '많음' 아니면 '적음'으로 조회

SELECT c.category_name, COUNT(p.product_id) AS 상품수,
    CASE
        WHEN COUNT(p.product_id) > 3 THEN '많음'
        ELSE '적음'
    END AS lvl
FROM tb_category c
    JOIN tb_product p ON c.category_id = p.category_id
GROUP BY c.category_id;

In [ ]:
-- Q17. 고객별 주문 건수를 0건 / 1~2건 / 3건 이상 구간으로 나눠 인원수 집계

SELECT 주문건수, COUNT(*) AS 인원수
FROM (
    SELECT c.customer_name, 
        CASE
            WHEN COUNT(o.order_id) = 0 THEN '0건'
            WHEN COUNT(o.order_id) BETWEEN 1 AND 2 THEN '1~2건'
            ELSE '3건 이상'
        END AS 주문건수
    FROM tb_customer c
        LEFT JOIN tb_order o ON c.customer_id = o.customer_id
    GROUP BY c.customer_id
) AS sq
GROUP BY 주문건수;

In [ ]:
-- Q18. 연도×월 매트릭스로 월별 매출 출력 (열: 1월~12월)

SELECT 연도,
    SUM(CASE WHEN 월=1 THEN 매출 ELSE 0 END) AS January,
    SUM(CASE WHEN 월=2 THEN 매출 ELSE 0 END) AS February,
    SUM(CASE WHEN 월=3 THEN 매출 ELSE 0 END) AS March,
    SUM(CASE WHEN 월=4 THEN 매출 ELSE 0 END) AS April,
    SUM(CASE WHEN 월=5 THEN 매출 ELSE 0 END) AS May,
    SUM(CASE WHEN 월=6 THEN 매출 ELSE 0 END) AS June,
    SUM(CASE WHEN 월=7 THEN 매출 ELSE 0 END) AS July,
    SUM(CASE WHEN 월=8 THEN 매출 ELSE 0 END) AS August,
    SUM(CASE WHEN 월=9 THEN 매출 ELSE 0 END) AS September,
    SUM(CASE WHEN 월=10 THEN 매출 ELSE 0 END) AS October,
    SUM(CASE WHEN 월=11 THEN 매출 ELSE 0 END) AS November,
    SUM(CASE WHEN 월=12 THEN 매출 ELSE 0 END) AS December
FROM (
    SELECT 
        YEAR(o.order_dt) AS 연도,
        MONTH(o.order_dt) AS 월,
        SUM(oi.qty * oi.unit_price) AS 매출
    FROM tb_order o
        JOIN tb_order_item oi ON o.order_id = oi.order_id
    WHERE o.status <> 'CANCELED'
    GROUP BY YEAR(o.order_dt), MONTH(o.order_dt)
) AS sq
GROUP BY 연도;


In [ ]:
-- [다중 조인]
-- Q19. 상품별 재고 소진율

SELECT cat.category_name, p.product_name, 
       IFNULL(SUM(oi.qty), 0) AS 판매수량, 
       p.stock_qty AS 남은재고,
       ((IFNULL(SUM(oi.qty), 0)) / (IFNULL(SUM(oi.qty), 0) + p.stock_qty)) * 100 AS 소진율
FROM tb_category cat
    JOIN tb_product p ON cat.category_id = p.category_id
    JOIN tb_order_item oi ON p.product_id = oi.product_id
GROUP BY p.product_id
ORDER BY 소진율 DESC;


In [ ]:
-- Q20. 취소로 놓친 매출

SELECT c.customer_name AS 고객명, 
    c.country AS 국가, 
    o.order_dt AS 주문일자, 
    oi.unit_price * oi.qty AS 취소금액, 
    count(*) AS 담긴상품개수
FROM tb_order o
    JOIN tb_order_item oi ON o.order_id = oi.order_id
    JOIN tb_customer c ON o.customer_id = c.customer_id
WHERE o.status = 'CANCELED'
GROUP BY o.order_id, c.customer_id, o.order_dt
ORDER BY 취소금액 DESC;

-- 취소 총액

SELECT SUM(취소금액) AS 취소총액
FROM (
    SELECT c.customer_name AS 고객명, 
        c.country AS 국가, 
        o.order_dt AS 주문일자, 
        oi.unit_price * oi.qty AS 취소금액, 
        count(*) AS 담긴상품개수
    FROM tb_order o
        JOIN tb_order_item oi ON o.order_id = oi.order_id
        JOIN tb_customer c ON o.customer_id = c.customer_id
    WHERE o.status = 'CANCELED'
    GROUP BY o.order_id, c.customer_id, o.order_dt
    ORDER BY 취소금액 DESC
) AS sq;

In [ ]:
-- Q21. 등급별 1인당 구매액

SELECT IFNULL(c.grade, '미지정') AS 등급, 
    COUNT(DISTINCT c.customer_id) AS 인원수,
    COUNT(DISTINCT o.customer_id) AS 주문한인원수,
    IFNULL(SUM(oi.unit_price * oi.qty), 0) AS 총매출,
    ROUND(IFNULL(SUM(oi.unit_price * oi.qty), 0) / COUNT(DISTINCT c.customer_id), 2) AS 1인당평균구매액
FROM tb_customer c
    LEFT JOIN tb_order o ON c.customer_id = o.customer_id
        AND o.status <> 'CANCELED'
    LEFT JOIN tb_order_item oi ON o.order_id = oi.order_id
GROUP BY IFNULL(c.grade, '미지정')
ORDER BY 1인당평균구매액 DESC;